# Training Grover's circuit on a simulated noisy chip (digital-qpu v0.17.0)

One shared set of angles is trained on **all 8 possible marked items at once** (average success), so it cannot memorise an answer - the v0.16.0 run did exactly that. Every trained circuit is also checked on the ideal chip for each marked item: if some answers are favoured (spread > 0.10) it does not count.

**Targets (set before running, EXPERIMENTS.md v0.17.0)**
1. Ideal chip: best valid trained mean >= 0.97 (fixed Grover 0.9453)
2. `dq-5` (training calibration): best valid trained mean >= fixed Grover mean + 0.03
3. 10 unseen calibration days: best valid trained minus the better fixed Grover (1 or 2 rounds) >= +0.02

Honest limits: classical simulation, no speedup; this optimises circuits for a simulated chip. On a phone it takes roughly 45-60 minutes; Colab is faster when it connects.

In [ ]:
!pip install -q "git+https://github.com/akosidave31/digital-qpu@v0.17.0"
import digital_qpu
print("digital-qpu", digital_qpu.__version__)

## 1. Sanity check: Grover is the starting point of the trainable circuit

In [ ]:
import time
from digital_qpu import DEVICES
from digital_qpu.variational import VariationalGrover, fixed_grover_success, joint_experiment, summarize_joint
for name in ("ideal", "dq-5"):
    dev = DEVICES[name]
    for rounds in (2, 1):
        vg = VariationalGrover(rounds=rounds)
        t0 = time.time()
        s = vg.success(vg.grover_init(), dev)
        print(f"{name:6} rounds={rounds}: trainable circuit at Grover point {s:.4f} | fixed Grover {fixed_grover_success(dev, rounds):.4f} | {time.time()-t0:.2f} s per evaluation")

## 2. Train and test (the whole experiment)
Epoch progress is printed every 5 epochs. If Colab disconnects, rerun this cell.

In [ ]:
t0 = time.time()
out = joint_experiment(epochs=40, lr=0.05, test_days=range(1, 11))
print(f"total {time.time()-t0:.0f} s")

## 3. Verdict against the targets

In [ ]:
for line in summarize_joint(out):
    print(line)

## 4. Training curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.5))
for key, run in out["runs"].items():
    ax.plot(run["history"], label=f"{key} (best {run['best']:.3f})")
ax.axhline(out["runs"]["dq-5/rounds2"]["start"], color="gray", ls="--", lw=1, label="fixed Grover on dq-5 (mean of 8 items)")
ax.set_xlabel("epoch"); ax.set_ylabel("P(marked answer)"); ax.set_ylim(0, 1.02); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## 5. Results to paste back (and a saved copy)

In [ ]:
import json
json.dump(out, open("grover_training_results.json", "w"), indent=1)
print("runs:")
for k, r in out["runs"].items():
    print(f"  {k:22} start {r['start']:.4f}  best {r['best']:.4f}  ({r['seconds']:.0f} s)")
print("unseen days (fixed Grover vs trained):")
for d, row in out["test"].items():
    print(f"  day {d:>2}: " + "  ".join(f"{k} {v:.3f}" for k, v in row.items()))